# ECG Classification — Colab Runner
Her session başında **sadece Bölüm 0 ve 1** hücrelerini çalıştırın.
Sonra istediğiniz deneyi Bölüm 2 den başlatın.

## Bölüm 0 — Kurulum (her session başında)

In [ ]:
!pip -q install wfdb iterative-stratification tqdm

import os, sys
REPO = "https://github.com/KULLANICI_ADINIZ/ecg_project.git"  # ← kendi URL
if not os.path.exists("/content/ecg_project"):
    !git clone {REPO} /content/ecg_project
else:
    !git -C /content/ecg_project pull

sys.path.insert(0, "/content/ecg_project")
print("✅ Kod hazır")

## Bölüm 1 — Drive mount + veri yükleme

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/ecg_outputs"

# npy dosyaları
if not os.path.exists("/content/PTBXL_records100_restore/npy_100"):
    !mkdir -p /content/PTBXL_records100_restore
    !tar -xf "{DRIVE}/npy_100.tar" -C /content/PTBXL_records100_restore/

# CV split
CV_SRC = f"{DRIVE}/cv_npy100_patientwise_superclass_k5"
if not os.path.exists("/content/cv_npy100_patientwise_superclass_k5"):
    !cp -r "{CV_SRC}" /content/

# Çıktı klasörleri
!mkdir -p "{DRIVE}/checkpoints" "{DRIVE}/reports"
print("✅ Veri hazır")

## Bölüm 2 — Deney çalıştır

In [ ]:
from config import list_presets
for p in list_presets(): print(p)

In [ ]:
from run_experiment import run

# End-to-end backbone:
df = run("resnet1d_100hz", epochs=20)

In [ ]:
# Frozen head (InceptionTime önce eğitilmiş olmalı):
BACKBONE = "/content/drive/MyDrive/ecg_outputs/checkpoints/inceptiontime_100hz"
# df = run("inception_frozen_kan_100hz", backbone_ckpt_dir=BACKBONE, epochs=10)

In [ ]:
# Partial fine-tune:
# df = run("inception_partial_ft_mlp_100hz", backbone_ckpt_dir=BACKBONE, epochs=20)

## Bölüm 3 — Sonuç karşılaştırması

In [ ]:
import json, glob, pandas as pd

summaries = []
for p in glob.glob("/content/drive/MyDrive/ecg_outputs/reports/*/summary.json"):
    with open(p) as f: summaries.append(json.load(f))

df_cmp = pd.DataFrame(summaries)
cols = ["experiment","test_macro_auc_mean","test_macro_auc_std",
        "test_macro_f1_mean","test_macro_f1_std"]
df_cmp[cols].sort_values("test_macro_auc_mean", ascending=False).round(4)

## Bölüm 4 — Label + CV split hazırlama (bir kez)

In [ ]:
# ptbxl_database.csv ve scp_statements.csv /content altında olmalı.
!python /content/ecg_project/scripts/prepare_labels.py \n    --root /content \n    --npy_dir /content/PTBXL_records100_restore/npy_100 \n    --out_dir /content/cv_npy100_patientwise_superclass_k5 \n    --sr 100

# Drive a yedekle
!cp -r /content/cv_npy100_patientwise_superclass_k5 \n        /content/drive/MyDrive/ecg_outputs/